Imports

In [227]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Gym Data Importation

In [228]:
data = pd.read_csv('gym_membership.csv', index_col = 0)

Gym Data Exploration

In [229]:
data.head()

,gender,birthday,Age,abonoment_type,visit_per_week,days_per_week,attend_group_lesson,fav_group_lesson,avg_time_check_in,avg_time_check_out,avg_time_in_gym,drink_abo,fav_drink,personal_training,name_personal_trainer,uses_sauna
id,,,,,,,,,,,,,,,,
1,Female,1997-04-18,27,Premium,4,"Mon, Sat, Tue, Wed",True,"Kickboxen, BodyPump, Zumba",19:31:00,21:27:00,116,False,NaN,False,NaN,True
2,Female,1977-09-18,47,Standard,3,"Mon, Sat, Wed",False,NaN,19:31:00,20:19:00,48,False,NaN,True,Chantal,False
3,Male,1983-03-30,41,Premium,1,Sat,True,XCore,08:29:00,10:32:00,123,True,"berry_boost, lemon",True,Mike,False
4,Male,1980-04-12,44,Premium,3,"Sat, Tue, Wed",False,NaN,09:54:00,11:33:00,99,True,passion_fruit,True,Mike,True
5,Male,1980-09-10,44,Standard,2,"Thu, Wed",True,"Running, Yoga, Zumba",08:29:00,09:19:00,50,False,NaN,True,Mike,False


Gym Data information

In [255]:
data.duplicated().sum()

np.int64(0)

In [230]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1 to 1000
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   gender                 1000 non-null   object
 1   birthday               1000 non-null   object
 2   Age                    1000 non-null   int64 
 3   abonoment_type         1000 non-null   object
 4   visit_per_week         1000 non-null   int64 
 5   days_per_week          1000 non-null   object
 6   attend_group_lesson    1000 non-null   bool  
 7   fav_group_lesson       503 non-null    object
 8   avg_time_check_in      1000 non-null   object
 9   avg_time_check_out     1000 non-null   object
 10  avg_time_in_gym        1000 non-null   int64 
 11  drink_abo              1000 non-null   bool  
 12  fav_drink              496 non-null    object
 13  personal_training      1000 non-null   bool  
 14  name_personal_trainer  518 non-null    object
 15  uses_sauna             100

In [231]:
data.describe()

,Age,visit_per_week,avg_time_in_gym
count,1000.000000,1000.000000,1000.000000
mean,30.604000,2.682000,105.260000
std,10.817958,1.241941,43.557177
min,12.000000,1.000000,30.000000
25%,21.000000,2.000000,67.000000
50%,30.000000,3.000000,104.000000
75%,40.000000,3.000000,143.000000
max,49.000000,5.000000,180.000000


Notes in which we have to transform this variables into:

- Gender - (0/1) , 1 as Male, 0 as Female
- Birthday - Encoding by generations
- Age - Drop (Redundande with the birthday)
- Subscription type - (0/1) , 1 as Premium, 0 as Standard
- Number of visits per week - Equal
- Which days on the week - Enconding
- Attend group lessons - Drop (Next columns will say if the client attend group lessons)
- Favortite group lesson - Encoding
- Average time check in - Encoding
- Average time check out - Drop (Redundance since we have the check in and average time in the gym)
- Average time in the gym - Equal
- Drink or not during the workout - Drop (Next columns will say if the client drinks something)
- Favorite drink - Encoding
- Has a personal trainer - Drop (Next columns will say if the client has a personal trainer)
- Name of the personal trainer - Encoding
- Uses sauna - (0/1) , 1 being 'Yes', 0 being 'No'

New and Clean Dataset (Improved)

In [232]:
improved = pd.DataFrame()

Gender

In [233]:
improved['Gender'] = data['gender'].map({'Male': 1, 'Female': 0})

Generation

- Baby Boomers: start 1946

- Generation X: start 1965

- Millennials (Generation Y): start 1981

- Generation Z: start 1997

- Generation Alpha: start 2010

- Generation Beta: start 2025 (projected)

In [234]:
improved['Baby Boomers'] = data['birthday'].dropna().apply(lambda i: 1 if int(i.split('-')[0]) >= 1946 and int(i.split('-')[0]) <= 1964 else 0)
improved['Generation X'] = data['birthday'].dropna().apply(lambda i: 1 if int(i.split('-')[0]) >= 1965 and int(i.split('-')[0]) <= 1980 else 0)
improved['Millennials'] = data['birthday'].dropna().apply(lambda i: 1 if int(i.split('-')[0]) >= 1981 and int(i.split('-')[0]) <= 1996 else 0)
improved['Generation Z'] = data['birthday'].dropna().apply(lambda i: 1 if int(i.split('-')[0]) >= 1997 and int(i.split('-')[0]) <= 2012 else 0)
improved['Generation Alpha'] = data['birthday'].dropna().apply(lambda i: 1 if int(i.split('-')[0]) >= 2013 else 0)

Subscription

In [235]:
improved['Subscription'] = data['abonoment_type'].map({'Premium': 1, 'Standard': 0})

Number of Visits per Week

In [236]:
improved['Number of Visits per Week'] = data['visit_per_week']

Which days of the week the client goes to the gym

In [237]:
days_of_the_week = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

for day in days_of_the_week:
    improved[day] = data['days_per_week'].str.contains(day, case=False, na=False).astype(int)

Which class the client attends

🥊 High Intensity

- Kickboxen
- XCore
- HIT

💃 Dance / Rhythm

- Zumba
- LesMiles

🏋️ Strength / Conditioning

- BodyPump
- Spinning

🧘 Mind / Body

- Yoga
- BodyBalance
- Pilates

🏃 Cardio

- Running

In [238]:
improved['High Intensity'] = 0
improved['Dance/Rhythm'] = 0
improved['Strength/Conditioning'] = 0
improved['Mind/Body'] = 0
improved['Cardio'] = 0


In [239]:
high_intensity = ['Kickboxen', 'XCore', 'HIT']
dance          = ['Zumba', 'LesMiles']
strength       = ['BodyPump', 'Spinning']
mind_body      = ['Yoga', 'BodyBalance', 'Pilates']
cardio         = ['Running']

for i in data.index:
    
    if pd.isna(data.loc[i,'fav_group_lesson']):
        continue
    
    lessons = [x.strip() for x in data.loc[i, 'fav_group_lesson'].split(',')]

    for lesson in lessons:
        if lesson in high_intensity:
            improved.loc[i, 'High Intensity'] = 1
        if lesson in dance:
            improved.loc[i, 'Dance/Rhythm'] = 1
        if lesson in strength:
            improved.loc[i, 'Strength/Conditioning'] = 1
        if lesson in mind_body:
            improved.loc[i, 'Mind/Body'] = 1
        if lesson in cardio:
            improved.loc[i, 'Cardio'] = 1   
        

Average time the client checks in

In [240]:
data['avg_time_check_in'] = data['avg_time_check_in'].astype('datetime64[ns]')
data['avg_time_check_out'] = data['avg_time_check_out'].astype('datetime64[ns]')

In [241]:
hours = data['avg_time_check_in'].dt.hour

improved['Morning'] = ((hours >= 0) & (hours < 12)).astype(int)
improved['Afternoon'] = ((hours >= 12) & (hours < 17)).astype(int)
improved['Evening'] = ((hours >= 17) & (hours < 24)).astype(int)

Average workout duration

In [242]:
improved['Average Workout Duration'] = data['avg_time_in_gym']

If the client drinks something during the workout and which

In [243]:
improved['Berry Boost'] = 0
improved['Lemon'] = 0
improved['Passion Fruit'] = 0
improved['Coconut Pineapple'] = 0
improved['Orange'] = 0
improved['Black Currant'] = 0

In [244]:
flavours = ['berry_boost', 'lemon', 'passion_fruit', 'coconut_pineapple', 'orange', 'black_currant']

for i in data.index:
    
    if pd.isna(data.loc[i, 'fav_drink']):
        continue

    drinks = [x.strip() for x in data.loc[i, 'fav_drink'].split(',')]
              
    if 'berry_boost' in drinks:
        improved.loc[i, 'Berry Boost'] = 1
    if 'lemon' in drinks:
        improved.loc[i, 'Lemon'] = 1
    if 'passion_fruit' in drinks:
        improved.loc[i, 'Passion Fruit'] = 1
    if 'coconut_pineapple' in drinks:
        improved.loc[i, 'Coconut Pineapple'] = 1
    if 'orange' in drinks:
        improved.loc[i, 'Orange'] = 1
    if 'black_currant' in drinks:
        improved.loc[i, 'Black Currant'] = 1

If the client has personal trainer and which

In [245]:
improved['Chantal'] = 0
improved['Mike'] = 0
improved['Jeffrey'] = 0
improved['Hanna'] = 0

In [246]:
names = ['Chantal', 'Mike', 'Jeffrey', 'Hanna']

for i in data.index:
    if pd.isna(data.loc[i, 'name_personal_trainer']):
        continue

    for name in names:
        if name in data.loc[i, 'name_personal_trainer']:
            improved.loc[i, name] = 1

If the client uses the sauna

In [247]:
improved['Sauna'] = data['uses_sauna'].map({True: 1, False: 0})

Save some space 

In [249]:
for i in improved.columns:
    if i == 'Average Workout Duration':
        improved[i] = improved[i].astype('Int16')
    if i == 'Number of Visits per Week':
        improved[i] = improved[i].astype('Int16')
    

    improved[i] = improved[i].astype('Int8')

Improved dataset information

In [250]:
improved.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1 to 1000
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Gender                     1000 non-null   Int8 
 1   Baby Boomers               1000 non-null   Int8 
 2   Generation X               1000 non-null   Int8 
 3   Millennials                1000 non-null   Int8 
 4   Generation Z               1000 non-null   Int8 
 5   Generation Alpha           1000 non-null   Int8 
 6   Subscription               1000 non-null   Int8 
 7   Number of Visits per Week  1000 non-null   Int8 
 8   Mon                        1000 non-null   Int8 
 9   Tue                        1000 non-null   Int8 
 10  Wed                        1000 non-null   Int8 
 11  Thu                        1000 non-null   Int8 
 12  Fri                        1000 non-null   Int8 
 13  Sat                        1000 non-null   Int8 
 14  Sun                        10

In [254]:
improved.describe()

,Gender,Baby Boomers,Generation X,Millennials,Generation Z,Generation Alpha,Subscription,Number of Visits per Week,Mon,Tue,...,Lemon,Passion Fruit,Coconut Pineapple,Orange,Black Currant,Chantal,Mike,Jeffrey,Hanna,Sauna
count,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,...,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0
mean,0.497,0.0,0.16,0.419,0.421,0.0,0.493,2.682,0.349,0.391,...,0.115,0.119,0.134,0.126,0.111,0.153,0.14,0.111,0.114,0.493
std,0.500241,0.0,0.366789,0.493642,0.493967,0.0,0.500201,1.241941,0.476892,0.488219,...,0.319182,0.323951,0.340823,0.332015,0.314289,0.360168,0.347161,0.314289,0.31797,0.500201
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,1.0,0.0,0.0,1.0,1.0,0.0,1.0,3.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
max,1.0,0.0,1.0,1.0,1.0,0.0,1.0,5.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Check if we have any missing values

In [251]:
improved.isnull().any().any()  # False = no missing values anywhere

np.False_

In [256]:
improved.to_csv('clean_gym_membership.csv', index=False)

In [252]:
lst = (data['birthday']
       .dropna()
       .astype(str)
       .str.split(',')
       .explode()
       .str.strip()
       .unique()
       .tolist())

In [253]:
lst

['1997-04-18',
 '1977-09-18',
 '1983-03-30',
 '1980-04-12',
 '1980-09-10',
 '2009-06-29',
 '1994-08-07',
 '2003-11-13',
 '1978-07-28',
 '2000-05-06',
 '1983-07-11',
 '1997-09-19',
 '1994-04-22',
 '1989-03-21',
 '1989-10-01',
 '1987-12-20',
 '1986-12-17',
 '2011-04-30',
 '1986-05-29',
 '1991-09-13',
 '1997-12-31',
 '1997-10-29',
 '2010-10-22',
 '2010-01-24',
 '2008-10-04',
 '1989-06-17',
 '2005-03-31',
 '1992-04-10',
 '1997-01-21',
 '2002-11-04',
 '1988-03-04',
 '2000-03-02',
 '1982-02-26',
 '1976-12-07',
 '2004-03-06',
 '1990-03-09',
 '1981-03-18',
 '1983-11-01',
 '1980-07-05',
 '2006-01-19',
 '2008-06-11',
 '1989-07-08',
 '1995-06-26',
 '1996-04-05',
 '1976-01-19',
 '1981-05-22',
 '1989-03-04',
 '1997-11-15',
 '1975-10-23',
 '1979-12-29',
 '1979-07-17',
 '1996-04-21',
 '2004-07-17',
 '1980-11-27',
 '2000-01-04',
 '2002-07-26',
 '2001-06-12',
 '1977-12-30',
 '2007-07-11',
 '1988-08-20',
 '1977-05-08',
 '1995-05-13',
 '1994-03-10',
 '2002-05-16',
 '1982-11-21',
 '1979-03-27',
 '2002-04-